# Interactive LLM Evaluation & Raw Sample Testing

This notebook provides an interactive environment to:
1. **Inspect Raw JSONL Run Logs**: Examine verbatim model output responses (`raw_response`), string lengths, expected vs extracted answers, and score reasons using the latest evaluator rules.
2. **Interactive Live Model Test**: Run single-sample inference dynamically on any configured model or benchmark.
3. **VRAM Cleanup**: Clear PyTorch CUDA GPU memory.

In [1]:
import sys
import json
from pathlib import Path

# Add project root to sys.path
WORKSPACE_ROOT = Path(".").resolve()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from eval import (
    load_models_config,
    resolve_model_path,
    load_model_and_tokenizer,
    run_batch_inference,
    discover_available_datasets,
    extract_think_part,
    extract_final_answer,
    clear_gpu_vram
)
from evaluators import get_evaluator

print("✓ Core evaluation modules loaded successfully.")

✓ Core evaluation modules loaded successfully.


## 1. Inspect Raw JSONL Run Logs
Load and verify exact `raw_response` values, character lengths, and scoring outputs using the latest evaluator rules.

In [2]:
# Specify the JSONL log file to inspect
log_path = WORKSPACE_ROOT / "outputs" / "run_20260729_154003" / "logs" / "Qwen2.5-0.5B-Instruct_mmlu_pro.jsonl"

if not log_path.exists():
    outputs_dir = WORKSPACE_ROOT / "outputs"
    run_dirs = sorted([d for d in outputs_dir.glob("run_*") if d.is_dir()], key=lambda x: x.stat().st_mtime, reverse=True)
    if run_dirs:
        logs = sorted(run_dirs[0].glob("logs/*.jsonl"))
        if logs:
            log_path = logs[0]

b_name = log_path.stem.split("_")[-1]
evaluator = get_evaluator(b_name)

print(f"[INSPECTING LOG FILE]: {log_path}")
print(f"[EVALUATOR MODULE ]: {b_name.upper()}")
print("=" * 80)

with open(log_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue
        entry = json.loads(line.strip())
        raw_resp = entry.get("raw_response", "")
        raw_item = entry.get("raw_item", {})
        
        clean_out = extract_final_answer(raw_resp)
        expected = evaluator.get_expected_answer(raw_item) if hasattr(evaluator, "get_expected_answer") else entry.get("expected_answer", "N/A")
        is_pass, pred_val, score_reason = evaluator.score_item(raw_item, clean_out)
        status = "✅ PASS" if is_pass else "❌ FAIL"
        
        print(f"[SAMPLE {entry.get('sample_index', i+1)}] Status: {status}")
        print(f"  - Expected Answer : '{expected}'")
        print(f"  - Extracted Answer: '{pred_val}'")
        print(f"  - Raw Response    : {repr(raw_resp)}")
        print(f"  - Raw Length      : {len(raw_resp)} character(s)")
        print(f"  - Score Reason    : {score_reason}")
        print("-" * 80)

[INSPECTING LOG FILE]: /home/common/Gen-AI/LLM-GenAI-Eval-Suite/outputs/run_20260729_154003/logs/Qwen2.5-0.5B-Instruct_mmlu_pro.jsonl
[EVALUATOR MODULE ]: PRO
[SAMPLE 1] Status: ✅ PASS
  - Expected Answer : 'I'
  - Extracted Answer: 'I'
  - Raw Response    : 'I) Safe practices, Distress, Fear, Serious'
  - Raw Length      : 42 character(s)
  - Score Reason    : Extracted 'I', Expected 'I'
--------------------------------------------------------------------------------
[SAMPLE 2] Status: ❌ FAIL
  - Expected Answer : 'F'
  - Extracted Answer: 'J'
  - Raw Response    : 'J) Stakeholder, Care and Skill, Diligence'
  - Raw Length      : 41 character(s)
  - Score Reason    : Extracted 'J', Expected 'F'
--------------------------------------------------------------------------------
[SAMPLE 3] Status: ✅ PASS
  - Expected Answer : 'J'
  - Extracted Answer: 'J'
  - Raw Response    : 'J) Down, Involvement, Remuneration, Compensation'
  - Raw Length      : 48 character(s)
  - Score Reason    : Ext

## 2. Interactive Live Model Inference
Select a model and benchmark to run live inference and inspect output tensors.

In [3]:
SELECTED_MODEL = "Qwen2.5-0.5B-Instruct"
SELECTED_BENCHMARK = "mmlu_pro"
SAMPLE_INDEX = 2  # 0-based index for Sample 3

resolved_path = resolve_model_path(SELECTED_MODEL)
data_file = WORKSPACE_ROOT / "data" / f"{SELECTED_BENCHMARK}.json"

if data_file.exists():
    with open(data_file, "r", encoding="utf-8") as f:
        items = json.load(f)
    
    item = items[SAMPLE_INDEX]
    evaluator = get_evaluator(SELECTED_BENCHMARK)
    prompt = evaluator.format_prompt(item)
    
    print(f"Loading model '{SELECTED_MODEL}' and running inference for Sample {SAMPLE_INDEX + 1}...")
    model, tokenizer, device = load_model_and_tokenizer(resolved_path)
    res = run_batch_inference(model, tokenizer, device, [prompt], max_new_tokens=512)[0]
    
    raw_out = res["text"]
    extracted_out = extract_final_answer(raw_out)
    think_out = extract_think_part(raw_out)
    is_pass, pred, reason = evaluator.score_item(item, extracted_out)
    
    print("\n" + "=" * 75)
    print(f"RAW UNFILTERED MODEL RESPONSE ({len(raw_out)} chars):")
    print("=" * 75)
    print(repr(raw_out))
    print("\n" + "=" * 75)
    print(f"EVALUATION RESULT: {'✅ PASS' if is_pass else '❌ FAIL'}")
    print(f"Reason: {reason}")
    print("=" * 75)
    clear_gpu_vram()

Loading model 'Qwen2.5-0.5B-Instruct' and running inference for Sample 3...


/home/mtk34308/miniconda3/envs/llm-eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[INFO] Loading model '/home/common/Gen-AI/LLM-GenAI-Eval-Suite/models/Qwen2.5-0.5B-Instruct' on cuda:0...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2556.59it/s]


tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  14582,     25,   2619,
            525,   1378,   1887,   4714,   5815,    448,  65892,  63001,     13,
          32671,     62,    374,    264,   1376,   4265,    438,   4152,    311,
            279,   1995,   4842,    315,    279,  26669,    432,    646,    387,
          18280,    429,   8256,    614,    264,   1290,    311,   1414,    421,
            807,    525,   1660,   1865,  47732,     13,  32671,     62,    374,
            264,   2086,   4265,     11,   7945,    279,  32671,    563,   6328,
            429,   8256,   5258,    979,  17113,   1007,    624,   3798,    510,
             32,      8,   6285,     11,   9460,  16974,     11,   4926,    359,
          20927,     11,  66350,    198,     33,      8,   6285,     11,    758,
          12536,   7830,    